# 01 — Data Cleaning / Preprocessing (Member 1)

Loads the Kaggle IMDb TV-shows dataset from `dataset/tv_shows.csv`, documents its real structure
(rows, columns, dtypes, missing values, duplicates), cleans it by **reusing `src/preprocessing.py`**
(no duplicate pipeline), produces the ML-ready `dataset/cleaned_tv_shows.csv`, verifies
zero target leakage, and demonstrates a reproducible train/test split.

> Note: the original raw Kaggle file is gitignored/absent from the repo; the dataset stored in
> `dataset/tv_shows.csv` is the Kaggle-derived data used by this project (already cleaned upstream),
> so the cleaning steps below are idempotent.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

import sys
sys.path.insert(0, str(Path.cwd()))  # allow importing src/ when run from repo root

from src.preprocessing import load_raw_data, clean_tv_shows_data, split_features_targets

DATA_PATH = "dataset/tv_shows.csv"
CLEAN_PATH = "dataset/cleaned_tv_shows.csv"
RANDOM_STATE = 42

## A. Data — inspect the actual CSV (do not assume columns)

In [2]:
df_raw = load_raw_data(DATA_PATH)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
print("\nColumn names:", list(df_raw.columns))
print("\nData types:"); print(df_raw.dtypes)
print("\nMissing values per column:"); print(df_raw.isna().sum())
print("\nDuplicate rows:", int(df_raw.duplicated().sum()))
df_raw.head()

Rows: 3000
Columns: 8

Column names: ['title', 'release_year', 'runtime_mins', 'certificate', 'primary_genre', 'key_cast', 'imdb_rating', 'success_category']

Data types:
title                object
release_year          int64
runtime_mins        float64
certificate          object
primary_genre        object
key_cast             object
imdb_rating         float64
success_category     object
dtype: object

Missing values per column:
title               0
release_year        0
runtime_mins        0
certificate         0
primary_genre       0
key_cast            0
imdb_rating         0
success_category    0
dtype: int64

Duplicate rows: 0


,title,release_year,runtime_mins,certificate,primary_genre,key_cast,imdb_rating,success_category
0,The Family Man,2019,42.0,Not Rated,Action,"Manoj Bajpayee, Samantha Akkineni, Priyamani, ...",8.8,High
1,Lucifer,2016,45.0,Not Rated,Crime,"Tom Ellis, Lauren German, Lesley-Ann Brandt, K...",8.1,High
2,The Handmaid's Tale,2017,42.0,Not Rated,Drama,"Elisabeth Moss, Yvonne Strahovski, Joseph Fien...",8.4,High
3,StartUp,2016,60.0,Not Rated,Crime,"Adam Brody, Edi Gathegi, Otmara Marrero, Krist...",8.0,High
4,Game of Thrones,2011,44.0,Not Rated,Action,"Emilia Clarke, Peter Dinklage, Kit Harington, ...",9.3,High


## B. Cleaning / preprocessing

Handled here (via `src/preprocessing.py`): missing values (median fill for numeric, genre/cast fill),
duplicate removal, dtype fixes (`release_year` -> int, `runtime_mins` -> float, `imdb_rating` -> numeric),
categorical handling (primary genre extraction, certificate), feature engineering
(`success_category` tiers from `bin_rating`), and target-leakage avoidance.

In [3]:
df_clean = clean_tv_shows_data(df_raw)

print("Clean shape:", df_clean.shape)
print("\nMissing after cleaning:"); print(df_clean.isna().sum())
print("\nDuplicate rows after cleaning:", int(df_clean.duplicated().sum()))
print("\nDtypes after cleaning:"); print(df_clean.dtypes)
print("\nSuccess tiers:", df_clean["success_category"].value_counts().to_dict())
df_clean.head()

Clean shape: (3000, 8)

Missing after cleaning:
title               0
release_year        0
runtime_mins        0
certificate         0
primary_genre       0
key_cast            0
imdb_rating         0
success_category    0
dtype: int64

Duplicate rows after cleaning: 0

Dtypes after cleaning:
title                object
release_year          int64
runtime_mins        float64
certificate          object
primary_genre        object
key_cast             object
imdb_rating         float64
success_category     object
dtype: object

Success tiers: {'Moderate': 1311, 'Low': 861, 'High': 828}


,title,release_year,runtime_mins,certificate,primary_genre,key_cast,imdb_rating,success_category
0,The Family Man,2019,42.0,Not Rated,Action,"Manoj Bajpayee, Samantha Akkineni, Priyamani, ...",8.8,High
1,Lucifer,2016,45.0,Not Rated,Crime,"Tom Ellis, Lauren German, Lesley-Ann Brandt, K...",8.1,High
2,The Handmaid's Tale,2017,42.0,Not Rated,Drama,"Elisabeth Moss, Yvonne Strahovski, Joseph Fien...",8.4,High
3,StartUp,2016,60.0,Not Rated,Crime,"Adam Brody, Edi Gathegi, Otmara Marrero, Krist...",8.0,High
4,Game of Thrones,2011,44.0,Not Rated,Action,"Emilia Clarke, Peter Dinklage, Kit Harington, ...",9.3,High


In [4]:
# Produce the clean ML-ready dataset (existing artifact path used by the train pipelines)
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Wrote {CLEAN_PATH} ({len(df_clean)} rows)")

# Sanity: output schema matches Member 1 contract
expected = ["title", "release_year", "runtime_mins", "certificate",
            "primary_genre", "key_cast", "imdb_rating", "success_category"]
assert list(df_clean.columns) == expected, "Unexpected schema"
print("Schema OK")

Wrote dataset/cleaned_tv_shows.csv (3000 rows)
Schema OK


## Zero-leakage check + reproducible train/test split

In [5]:
# Leakage check: target and forbidden post-release columns must not be features
X, y_reg, y_class = split_features_targets(df_clean)
forbidden = {"imdb_rating", "no_of_votes", "Popularity_Rank", "User_Reviews", "success_category"}
leaked = forbidden.intersection(X.columns)
assert not leaked, f"Target leakage: {leaked}"
print("Leakage check PASSED. Feature columns:", list(X.columns))
print("Regression target y:", y_reg.name, "| Classification target:", y_class.name)

# Reproducible 80/20 split (same seed used by src/regression.py)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_reg, test_size=0.2, random_state=RANDOM_STATE)
X_tr2, X_te2, _, _ = train_test_split(X, y_reg, test_size=0.2, random_state=RANDOM_STATE)
assert X_tr.index.equals(X_tr2.index), "Split not reproducible"
print(f"Train: {len(X_tr)} | Test: {len(X_te)} | reproducible with random_state={RANDOM_STATE}")

Leakage check PASSED. Feature columns: ['release_year', 'runtime_mins', 'certificate', 'primary_genre', 'key_cast']
Regression target y: imdb_rating | Classification target: success_category
Train: 2400 | Test: 600 | reproducible with random_state=42
